# Evaluation: Austrian Tax Law LLM Comparison

This notebook evaluates three LLM approaches on the Austrian Tax Law dataset:
- **Inference**: Llama 3.3 70B via Groq API (zero-shot)
- **Fine-tuning**: Llama 3.1 8B fine-tuned with QLoRA
- **RAG**: FAISS + paraphrase-multilingual-MiniLM-L12-v2

**Metrics:**
- **ROUGE-L**: measures longest common subsequence overlap between generated and reference answers
- **BERTScore**: measures semantic similarity using multilingual BERT embeddings

Reference answers come from the shared course dataset (`correct_answer` column).

**Note on ROUGE-L:** Model answers are longer and more detailed than the short reference answers. ROUGE-L penalizes verbosity even when the answer is correct. BERTScore is therefore a more appropriate metric for this task.

**Note on the dataset:** The shared dataset contains a known ID issue in the VAT-INTL category, where all 82 entries share the ID `VAT-INTL-001`. This is a data quality issue in the shared dataset, not in the model outputs.

## Step 1: Install dependencies

In [ ]:
pip install pandas rouge-score bert-score

## Step 2: Configuration

Place this notebook in the same folder as the four CSV files and the paths will work as-is.

In [ ]:
GT_PATH        = "Austrian Tax Law LLM Prompt Assignment - Dataset.csv"
INFERENCE_PATH = "/Users/edwarddimapilis/Desktop/Uni/sbwl/Data Science/Kurs 4/project/12309335_Dimapilis_Edward/evaluation/inference_submission.csv"
FT_PATH        = "/Users/edwarddimapilis/Desktop/Uni/sbwl/Data Science/Kurs 4/project/12309335_Dimapilis_Edward/evaluation/ft_submission.csv"
RAG_PATH       = "/Users/edwarddimapilis/Desktop/Uni/sbwl/Data Science/Kurs 4/project/12309335_Dimapilis_Edward/evaluation/rag_submission.csv"

## Step 3: Load data

In [ ]:
import pandas as pd

gt     = pd.read_csv(GT_PATH)[['id', 'correct_answer']]
inf_df = pd.read_csv(INFERENCE_PATH)
ft_df  = pd.read_csv(FT_PATH)
rag_df = pd.read_csv(RAG_PATH)

inf_merged = gt.merge(inf_df.rename(columns={'answer': 'inference'}), on='id', how='inner')
ft_merged  = gt.merge(ft_df.rename(columns={'answer': 'ft'}),        on='id', how='inner')
rag_merged = gt.merge(rag_df.rename(columns={'answer': 'rag'}),      on='id', how='inner')

print(f"Inference rows: {len(inf_merged)}")
print(f"Fine-tuning rows: {len(ft_merged)}")
print(f"RAG rows: {len(rag_merged)}")

## Step 4: Clean answers

In [ ]:
def clean(series):
    return series.fillna('').astype(str)

ref_inf = clean(inf_merged['correct_answer']).tolist()
hyp_inf = clean(inf_merged['inference']).tolist()

ref_ft  = clean(ft_merged['correct_answer']).tolist()
hyp_ft  = clean(ft_merged['ft']).tolist()

ref_rag = clean(rag_merged['correct_answer']).tolist()
hyp_rag = clean(rag_merged['rag']).tolist()

print("Data ready.")

## Step 5: Compute ROUGE-L

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)

def compute_rouge_l(hypotheses, references):
    scores = [scorer.score(ref, hyp)['rougeL'].fmeasure
              for hyp, ref in zip(hypotheses, references)]
    return sum(scores) / len(scores)

rouge_inf = compute_rouge_l(hyp_inf, ref_inf)
rouge_ft  = compute_rouge_l(hyp_ft,  ref_ft)
rouge_rag = compute_rouge_l(hyp_rag, ref_rag)

print(f"ROUGE-L  Inference:   {rouge_inf:.4f}")
print(f"ROUGE-L  Fine-tuning: {rouge_ft:.4f}")
print(f"ROUGE-L  RAG:         {rouge_rag:.4f}")

## Step 6: Compute BERTScore

BERTScore uses `bert-base-multilingual-cased` to measure semantic similarity. The model is downloaded on first run (~700MB) and computation takes a few minutes on CPU.

In [ ]:
from bert_score import score as bert_score_fn

def compute_bertscore(hypotheses, references):
    P, R, F1 = bert_score_fn(
        hypotheses,
        references,
        model_type='bert-base-multilingual-cased',
        lang='de',
        verbose=False
    )
    return F1.mean().item()

print("Computing BERTScore for Inference...")
bert_inf = compute_bertscore(hyp_inf, ref_inf)

print("Computing BERTScore for Fine-tuning...")
bert_ft  = compute_bertscore(hyp_ft, ref_ft)

print("Computing BERTScore for RAG...")
hyp_rag = [h[:512] if h else "keine Antwort" for h in hyp_rag]
bert_rag = compute_bertscore(hyp_rag, ref_rag)

print(f"\nBERTScore (F1)  Inference:   {bert_inf:.4f}")
print(f"BERTScore (F1)  Fine-tuning: {bert_ft:.4f}")
print(f"BERTScore (F1)  RAG:         {bert_rag:.4f}")

## Step 7: Results table

In [ ]:
results = pd.DataFrame({
    'Model':           ['Inference (Llama 3.3 70B)', 'Fine-tuning (Llama 3.1 8B)', 'RAG'],
    'ROUGE-L':         [rouge_inf, rouge_ft, rouge_rag],
    'BERTScore (F1)':  [bert_inf,  bert_ft,  bert_rag]
})

results['ROUGE-L']        = results['ROUGE-L'].map('{:.4f}'.format)
results['BERTScore (F1)'] = results['BERTScore (F1)'].map('{:.4f}'.format)

print(results.to_string(index=False))

## Step 8: Error analysis

We look at the 5 worst-performing examples per model according to ROUGE-L to understand what kinds of errors each model makes.

In [ ]:
gt_full = pd.read_csv(GT_PATH)[['id', 'prompt']]

def get_per_sample_rouge(hypotheses, references):
    return [scorer.score(ref, hyp)['rougeL'].fmeasure
            for hyp, ref in zip(hypotheses, references)]

inf_merged['rouge_l'] = get_per_sample_rouge(hyp_inf, ref_inf)
ft_merged['rouge_l']  = get_per_sample_rouge(hyp_ft,  ref_ft)
rag_merged['rouge_l'] = get_per_sample_rouge(hyp_rag, ref_rag)

def show_worst(merged_df, answer_col, model_name, n=5):
    worst = merged_df.nsmallest(n, 'rouge_l').merge(gt_full, on='id', how='left')
    print(f"\n=== {model_name}: {n} worst answers by ROUGE-L ===")
    for _, row in worst.iterrows():
        print(f"\nID: {row['id']}")
        print(f"Question:  {str(row['prompt'])[:120]}")
        print(f"Reference: {str(row['correct_answer'])[:200]}")
        print(f"Generated: {str(row[answer_col])[:200]}")
        print(f"ROUGE-L:   {row['rouge_l']:.4f}")
        print("-" * 60)

show_worst(inf_merged, 'inference', 'Inference')
show_worst(ft_merged,  'ft',        'Fine-tuning')
show_worst(rag_merged, 'rag',       'RAG')

## Step 9: Save results

In [ ]:
results.to_csv('evaluation_results.csv', index=False)
print("Saved to evaluation_results.csv")